# NBA 下赛季 PER 预测 —— XGBoost 出场时间样本加权优化

基于 `03_prediction_advance_plus.ipynb` 的回归流程，在相同的特征工程与时间切分基础上，引入 **赛季累计出场时间 `MP` 作为 `sample_weight`**，让模型训练时更关注主力球员。

优化点：
1. `time_split` 同步切分出训练集出场时间权重 `sample_weight_train`；
2. 通过 `Pipeline.fit(X, y, model__sample_weight=sample_weight_train)` 将权重仅传给 `XGBRegressor`；
3. 与加权前测试指标（R²=0.3760、RMSE=4.6607）进行同口径对比。

> 本 notebook 仅关注回归任务，不涉及 SHAP 与模型导出。

## 1. 数据准备与特征工程（沿用 `advance_plus`）

### 1.1 导入依赖库

In [1]:
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

print(f"pandas       : {pd.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"xgboost      : {xgb.__version__}")

pandas       : 2.3.3
scikit-learn : 1.9.0
xgboost      : 2.1.4


### 1.2 读取主数据集

In [2]:
DATA_PATH = "../data/processed/master_data.csv"

df_raw = pd.read_csv(DATA_PATH)

print(f"主数据集规模: {df_raw.shape[0]} 行 x {df_raw.shape[1]} 列")
print(f"赛季覆盖范围: {df_raw['Year'].min()} - {df_raw['Year'].max()}")

主数据集规模: 20313 行 x 78 列
赛季覆盖范围: 1950 - 2017


### 1.3 构造回归标签与新增时序特征

沿用 `advance_plus` 的断档保护逻辑：`PER_diff` 仅在相邻赛季（`Year` 差 == 1）时计算；新秀首季的 `PER_diff` 设为 0。

In [3]:
# 先按 (Player, Year) 排序，保证组内计算严格按时间先后进行
df = df_raw.sort_values(["Player", "Year"]).reset_index(drop=True)

# 回归标签：下赛季 PER
df["PER_next"] = df.groupby("Player")["PER"].shift(-1)

# ---- 1) PER_diff：仅计算相邻赛季的差值，断档置 NaN ----
prev_year = df.groupby("Player")["Year"].shift(1)
prev_PER = df.groupby("Player")["PER"].shift(1)
df["PER_diff"] = np.where(
    df["Year"] - prev_year == 1,
    df["PER"] - prev_PER,
    np.nan,
)

# ---- 2) Career_Year：职业生涯第几年 ----
df["Career_Year"] = (
    df["Year"] - df.groupby("Player")["Year"].transform("min") + 1
)

# 新秀首季没有“上一赛季”，趋势差语义上应为 0
df.loc[df["Career_Year"] == 1, "PER_diff"] = 0.0

# ---- 3) PER_3yr_avg：近三年 PER 移动平均 ----
df["PER_3yr_avg"] = df.groupby("Player")["PER"].transform(
    lambda s: s.rolling(3, min_periods=1).mean()
)

print("新增特征后的缺失情况：")
print(df[["PER_diff", "Career_Year", "PER_3yr_avg"]].isna().sum().to_string())

新增特征后的缺失情况：
PER_diff       976
Career_Year      0
PER_3yr_avg    367


### 1.4 剔除无关字段、Pos 独热编码并扩展 `feature_cols`

In [4]:
# 与先前 notebook 保持一致的剔除字段
drop_cols = [
    "Player",
    "Tm",
    "college",
    "birth_year",
    "birth_city",
    "birth_state",
    "birth_date",
    "year_start",
    "year_end",
    "position_career",
]

df_model = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

feature_cols = [
    # ---- 原有当季统计特征 ----
    "Age",
    "G",
    "MP",
    "TS%",
    "3PAr",
    "FTr",
    "USG%",
    "PTS_per36",
    "AST%",
    "TRB%",
    "WS",
    # ---- 新增时序 / 生涯特征 ----
    "PER_diff",
    "Career_Year",
    "PER_3yr_avg",
]

# Pos -> One-Hot Encoding
pos_dummies = pd.get_dummies(df_model["Pos"], prefix="Pos").astype(int)
pos_cols = pos_dummies.columns.tolist()

df_model = pd.concat(
    [df_model.drop(columns=["Pos"]), pos_dummies],
    axis=1,
)


def build_X(data: pd.DataFrame) -> pd.DataFrame:
    """构造回归特征矩阵（统计特征 + 时序特征 + Pos 哑变量）。"""
    return data[feature_cols + pos_cols].copy()


print(f"扩展后统计/时序特征数: {len(feature_cols)}")
print(f"Pos 哑变量数         : {len(pos_cols)}")

扩展后统计/时序特征数: 14
Pos 哑变量数         : 23


## 2. 回归样本与时间切分（同步切出 `sample_weight`）

### 2.1 构造回归样本 `X_reg` / `y_reg` 与出场时间权重 `MP`

In [5]:
# 删除无法匹配到下赛季 PER 的记录（最后一季）
reg_df = df_model[df_model["PER_next"].notna()].copy()

X_reg = build_X(reg_df)
y_reg = reg_df["PER_next"]
year_reg = reg_df["Year"]

# 样本权重：球员当季累计出场时间 MP（作为 sample_weight）
mp_weight = reg_df["MP"]

print(f"回归样本数: {X_reg.shape[0]}")
print(f"X_reg 维度 : {X_reg.shape[0]} 行 x {X_reg.shape[1]} 列")
print()
print("训练用 MP 权重统计:")
print(mp_weight.describe().to_string())
print(f"MP == 0 的样本数: {int((mp_weight == 0).sum())}")

回归样本数: 16260
X_reg 维度 : 16260 行 x 37 列

训练用 MP 权重统计:
count    16260.000000
mean      1536.963038
std        908.721503
min          0.000000
25%        770.000000
50%       1547.000000
75%       2288.000000
max       3882.000000
MP == 0 的样本数: 99


### 2.2 `time_split` 同步切分 `sample_weight_train`

保持原有规则：Train `Year <= 2010`、Test `Year > 2010`，并让权重与样本按同一掩码切分，避免索引错位。

In [6]:
def time_split(X, y, years, sample_weight=None, train_max=2010):
    """按赛季年份切分；若传入 sample_weight，则同步返回 train/test 权重。"""
    train_mask = years <= train_max
    test_mask = ~train_mask
    result = (
        X.loc[train_mask],
        X.loc[test_mask],
        y.loc[train_mask],
        y.loc[test_mask],
    )
    if sample_weight is not None:
        result += (
            sample_weight.loc[train_mask],
            sample_weight.loc[test_mask],
        )
    return result


(
    X_train_reg,
    X_test_reg,
    y_train_reg,
    y_test_reg,
    sample_weight_train,
    sample_weight_test,
) = time_split(X_reg, y_reg, year_reg, sample_weight=mp_weight)

print("训练集:", X_train_reg.shape, "| 测试集:", X_test_reg.shape)
print()
print("sample_weight_train 统计:")
print(sample_weight_train.describe().to_string())
print(f"sample_weight_train == 0 样本数: {int((sample_weight_train == 0).sum())}")

训练集: (13883, 37) | 测试集: (2377, 37)

sample_weight_train 统计:
count    13883.000000
mean      1563.397969
std        922.385929
min          0.000000
25%        782.000000
50%       1579.000000
75%       2337.000000
max       3882.000000
sample_weight_train == 0 样本数: 98


## 3. XGBoost 模型训练：引入 `MP` 样本加权

### 3.1 Pipeline 中正确传递 `sample_weight`

Pipeline 中各步骤通过 `<步骤名>__<参数名>` 语法传参。`sample_weight` 应只传给 XGBoost 拟合步骤：

```python
xgb_pipeline.fit(
    X_train_reg,
    y_train_reg,
    model__sample_weight=sample_weight_train,
)
```

`SimpleImputer` 的填补仍按常规中位数策略进行，不接收样本权重。

In [7]:
xgb_weighted_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    )),
])

# 关键点：sample_weight 通过 model__sample_weight 传递给 XGBRegressor
xgb_weighted_pipeline.fit(
    X_train_reg,
    y_train_reg,
    model__sample_weight=sample_weight_train,
)

y_pred_weighted = xgb_weighted_pipeline.predict(X_test_reg)

r2_weighted = r2_score(y_test_reg, y_pred_weighted)
rmse_weighted = float(np.sqrt(mean_squared_error(y_test_reg, y_pred_weighted)))

print("[回归] XGBRegressor (MP 样本加权后)")
print(f"  R2   = {r2_weighted:.4f}")
print(f"  RMSE = {rmse_weighted:.4f}")

[回归] XGBRegressor (MP 样本加权后)
  R2   = 0.3799
  RMSE = 4.6463


### 3.2 与样本加权前对比

加权前指标（R²=0.3760、RMSE=4.6607）来自 `03_prediction_advance_plus.ipynb` 的最终测试集结果，评估口径完全一致。

In [8]:
comparison_weighted = pd.DataFrame([
    {
        "Model": "XGBRegressor (MP 加权前)",
        "R2": 0.3760,
        "RMSE": 4.6607,
    },
    {
        "Model": "XGBRegressor (MP 加权后)",
        "R2": r2_weighted,
        "RMSE": rmse_weighted,
    },
])

print("出场时间样本加权前后对比表:")
print(comparison_weighted.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()

r2_gain = r2_weighted - 0.3760
rmse_gain = 4.6607 - rmse_weighted
print(f"R2 变化  : {r2_gain:+.4f}")
print(f"RMSE 变化: {rmse_gain:+.4f} (正值表示 RMSE 降低)")

if r2_gain > 0 and rmse_gain > 0:
    print("结论：MP 样本加权带来了测试集泛化性能的提升。")
else:
    print("结论：MP 样本加权未带来测试集性能提升，需进一步评估。")

出场时间样本加权前后对比表:
                Model     R2   RMSE
XGBRegressor (MP 加权前) 0.3760 4.6607
XGBRegressor (MP 加权后) 0.3799 4.6463

R2 变化  : +0.0039
RMSE 变化: +0.0144 (正值表示 RMSE 降低)
结论：MP 样本加权带来了测试集泛化性能的提升。


## 4. 阶段小结

- 以球员当季累计出场时间 `MP` 作为 `sample_weight`，让模型在训练时更重视出场时间更长的主力球员；
- 通过 `time_split` 同步切分权重，并使用 `model__sample_weight` 正确传入 Pipeline 中的 XGBoost；
- 在同一测试集（`Year > 2010`）上与加权前的 R² / RMSE 做同口径对比，验证样本加权是否改善泛化。

> 当前仅完成回归任务与指标输出，未涉及 SHAP 归因与模型导出。